In [1]:
import os
import cv2 
import dlib 
import keras
import imutils
import numpy as np 
import pandas as pd 
from imutils import face_utils
from keras import regularizers
from keras.preprocessing import image
from keras.models import Model, Sequential
from keras.preprocessing.image import ImageDataGenerator
from keras.applications.mobilenet import preprocess_input
from keras.layers import Dense, GlobalAveragePooling2D, MaxPooling2D, Conv2D, Dropout, Flatten

In [2]:
def rect_to_bb(rect):
    # take a bounding predicted by dlib and convert it
    # to the format (x, y, w, h) as we would normally do
    # with OpenCV
	x = rect.left()
	y = rect.top() 
	w = rect.right() - x
	h = rect.bottom() - y

    # return a tuple of (x, y, w, h)
	return (x, y, w, h)

In [5]:
def shape_to_np(shape, dtype="int"):
	# initialize the list of (x, y)-coordinates
	coords = np.zeros((68, 2), dtype=dtype)

	# loop over the 68 facial landmarks and convert them
	# to a 2-tuple of (x, y)-coordinates
	for i in range(0, 68):
		coords[i] = (shape.part(i).x, shape.part(i).y)

	# return the list of (x, y)-coordinates
	return coords

In [6]:
def crop_and_save_image(img, img_path, write_img_path, img_name):
	detector = dlib.get_frontal_face_detector()
	predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')
	# load the input image, resize it, and convert it to grayscale

	image = cv2.imread(img_path)
	gray=image
	# detect faces in the grayscale image
	rects = detector(gray, 1)
	if len(rects) > 1:
		print( "ERROR: more than one face detected")
		return
	if len(rects) < 1:
		print( "ERROR: no faces detected")
		return

	for (i, rect) in enumerate(rects):
		shape = predictor(gray, rect)
		shape = face_utils.shape_to_np(shape)
		name, i, j = 'mouth', 48, 68
		# clone = gray.copy()

		(x, y, w, h) = cv2.boundingRect(np.array([shape[i:j]]))
		print(x,y,w,h)
		padding_h=50-h
		padding_w=50-w
		if padding_h%2 ==1:
			padding_h_up=int(padding_h/2)
			padding_h_down=padding_h_up+1
		else:
			padding_h_up=int(padding_h/2)
			padding_h_down=padding_h_up
		y=y-padding_h_up
		h=h+padding_h_down

		if padding_w%2 ==1:
			padding_w_left=int(padding_w/2)
			padding_w_right=padding_w_left+1
		else:
			padding_w_left=int(padding_w/2)
			padding_w_right=padding_w_left
		x=x-padding_w_left
		w=w+padding_w_right
		w=h=50
		print(x,y,w,h)
		roi = gray[y:y+h, x:x+w]
		roi = cv2.resize(roi, (100,100),interpolation = cv2.INTER_AREA)        
		print('DATASET/colored cropped/' + write_img_path)
		cv2.imwrite('DATASET/colored cropped/' + write_img_path, roi)
		break	

In [7]:
os.listdir('DATASET')
predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')

In [8]:
people = ['F01','F02','F05','F04','F06','F07','F08','F09','F10','F11','M01','M02','M04','M07','M08']
data_types = ['words']
folder_enum = ['01','02','03','04','05','06','07','08', '09', '10']
instances = ['01','02','03','04','05','06','07','08', '09', '10']

In [9]:
words = ['Begin', 'Choose', 'Connection', 'Navigation', 'Next', 'Previous', 'Start', 'Stop', 'Hello', 'Web'] 
#words=['Begin', 'Choose']
words_di = {i:words[i] for i in range(len(words))}

In [10]:
if not os.path.exists('DATASET/colored cropped'):
    os.mkdir('DATASET/colored cropped')


In [11]:
for person_ID in people:
	if not os.path.exists('DATASET/colored cropped/' + person_ID ):
		os.mkdir('DATASET/colored cropped/' + person_ID)

	for data_type in data_types:
		if not os.path.exists('DATASET/colored cropped/' + person_ID + '/' + data_type):
			os.mkdir('DATASET/colored cropped/' + person_ID + '/' + data_type)

		for phrase_ID in folder_enum:
			if not os.path.exists('DATASET/colored cropped/' + person_ID + '/' + data_type + '/' + phrase_ID):
		         # F01/phrases/01
				os.mkdir('DATASET/colored cropped/' + person_ID + '/' + data_type + '/' + phrase_ID)
			for instance_ID in instances:
				directory = 'DATASET/' + person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID + '/'
				dir_temp = person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID + '/'
				print(directory)
				filelist = os.listdir(directory)
				if not os.path.exists('DATASET/colored cropped/' + person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID):
					os.mkdir('DATASET/colored cropped/' + person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID)
				for img_name in filelist:
					if img_name.startswith('color'):
						image = cv2.imread(directory + '' + img_name)
						crop_and_save_image(image, directory + '' + img_name,dir_temp + '' + img_name, img_name)

DATASET/F01/words/01/01/
225 360 28 11
214 341 50 50
DATASET/colored cropped/F01/words/01/01/color_001.jpg
225 359 28 15
214 342 50 50
DATASET/colored cropped/F01/words/01/01/color_002.jpg
224 358 30 18
214 342 50 50
DATASET/colored cropped/F01/words/01/01/color_003.jpg
225 363 29 7
215 342 50 50
DATASET/colored cropped/F01/words/01/01/color_004.jpg
224 358 32 16
215 341 50 50
DATASET/colored cropped/F01/words/01/01/color_005.jpg
222 355 38 18
216 339 50 50
DATASET/colored cropped/F01/words/01/01/color_006.jpg
223 355 35 19
216 340 50 50
DATASET/colored cropped/F01/words/01/01/color_007.jpg
225 356 30 16
215 339 50 50
DATASET/colored cropped/F01/words/01/01/color_008.jpg
225 356 30 13
215 338 50 50
DATASET/colored cropped/F01/words/01/01/color_009.jpg
225 357 29 11
215 338 50 50
DATASET/colored cropped/F01/words/01/01/color_010.jpg
DATASET/F01/words/01/02/
225 359 29 11
215 340 50 50
DATASET/colored cropped/F01/words/01/02/color_001.jpg
224 355 34 19
216 340 50 50
DATASET/colored cropp

224 352 28 15
213 335 50 50
DATASET/colored cropped/F01/words/02/02/color_001.jpg
225 350 26 20
213 335 50 50
DATASET/colored cropped/F01/words/02/02/color_002.jpg
226 350 23 18
213 334 50 50
DATASET/colored cropped/F01/words/02/02/color_003.jpg
227 350 21 18
213 334 50 50
DATASET/colored cropped/F01/words/02/02/color_004.jpg
225 351 27 15
214 334 50 50
DATASET/colored cropped/F01/words/02/02/color_005.jpg
223 351 32 16
214 334 50 50
DATASET/colored cropped/F01/words/02/02/color_006.jpg
225 351 29 15
215 334 50 50
DATASET/colored cropped/F01/words/02/02/color_007.jpg
224 354 29 9
214 334 50 50
DATASET/colored cropped/F01/words/02/02/color_008.jpg
DATASET/F01/words/02/03/
223 352 28 14
212 334 50 50
DATASET/colored cropped/F01/words/02/03/color_001.jpg
224 352 29 17
214 336 50 50
DATASET/colored cropped/F01/words/02/03/color_002.jpg
225 350 26 19
213 335 50 50
DATASET/colored cropped/F01/words/02/03/color_003.jpg
226 351 23 18
213 335 50 50
DATASET/colored cropped/F01/words/02/03/color_

228 354 27 13
217 336 50 50
DATASET/colored cropped/F01/words/03/04/color_001.jpg
229 353 25 18
217 337 50 50
DATASET/colored cropped/F01/words/03/04/color_002.jpg
228 354 26 15
216 337 50 50
DATASET/colored cropped/F01/words/03/04/color_003.jpg
227 353 30 17
217 337 50 50
DATASET/colored cropped/F01/words/03/04/color_004.jpg
227 351 30 17
217 335 50 50
DATASET/colored cropped/F01/words/03/04/color_005.jpg
228 353 28 16
217 336 50 50
DATASET/colored cropped/F01/words/03/04/color_006.jpg
227 354 29 11
217 335 50 50
DATASET/colored cropped/F01/words/03/04/color_007.jpg
DATASET/F01/words/03/05/
227 354 28 12
216 335 50 50
DATASET/colored cropped/F01/words/03/05/color_001.jpg
228 353 26 18
216 337 50 50
DATASET/colored cropped/F01/words/03/05/color_002.jpg
229 353 24 21
216 339 50 50
DATASET/colored cropped/F01/words/03/05/color_003.jpg
229 354 24 18
216 338 50 50
DATASET/colored cropped/F01/words/03/05/color_004.jpg
226 353 30 18
216 337 50 50
DATASET/colored cropped/F01/words/03/05/color

220 352 30 11
210 333 50 50
DATASET/colored cropped/F01/words/04/04/color_009.jpg
DATASET/F01/words/04/05/
220 352 29 12
210 333 50 50
DATASET/colored cropped/F01/words/04/05/color_001.jpg
221 351 28 17
210 335 50 50
DATASET/colored cropped/F01/words/04/05/color_002.jpg
220 352 32 21
211 338 50 50
DATASET/colored cropped/F01/words/04/05/color_003.jpg
220 350 30 18
210 334 50 50
DATASET/colored cropped/F01/words/04/05/color_004.jpg
220 351 31 17
211 335 50 50
DATASET/colored cropped/F01/words/04/05/color_005.jpg
220 352 32 17
211 336 50 50
DATASET/colored cropped/F01/words/04/05/color_006.jpg
220 350 29 18
210 334 50 50
DATASET/colored cropped/F01/words/04/05/color_007.jpg
220 352 30 14
210 334 50 50
DATASET/colored cropped/F01/words/04/05/color_008.jpg
220 353 30 10
210 333 50 50
DATASET/colored cropped/F01/words/04/05/color_009.jpg
DATASET/F01/words/04/06/
220 353 29 11
210 334 50 50
DATASET/colored cropped/F01/words/04/06/color_001.jpg
221 354 28 16
210 337 50 50
DATASET/colored crop

237 360 30 14
227 342 50 50
DATASET/colored cropped/F01/words/05/06/color_005.jpg
237 360 29 11
227 341 50 50
DATASET/colored cropped/F01/words/05/06/color_006.jpg
DATASET/F01/words/05/07/
236 361 29 12
226 342 50 50
DATASET/colored cropped/F01/words/05/07/color_001.jpg
237 360 29 14
227 342 50 50
DATASET/colored cropped/F01/words/05/07/color_002.jpg
235 358 35 18
228 342 50 50
DATASET/colored cropped/F01/words/05/07/color_003.jpg
234 358 37 17
228 342 50 50
DATASET/colored cropped/F01/words/05/07/color_004.jpg
234 358 37 17
228 342 50 50
DATASET/colored cropped/F01/words/05/07/color_005.jpg
235 359 36 17
228 343 50 50
DATASET/colored cropped/F01/words/05/07/color_006.jpg
237 359 30 15
227 342 50 50
DATASET/colored cropped/F01/words/05/07/color_007.jpg
236 361 31 10
227 341 50 50
DATASET/colored cropped/F01/words/05/07/color_008.jpg
DATASET/F01/words/05/08/
235 361 31 12
226 342 50 50
DATASET/colored cropped/F01/words/05/08/color_001.jpg
234 360 35 17
227 344 50 50
DATASET/colored crop

...

288 337 26 9
276 317 50 50
DATASET/colored cropped/M08/words/05/06/color_008.jpg
DATASET/M08/words/05/07/
288 337 26 10
276 317 50 50
DATASET/colored cropped/M08/words/05/07/color_001.jpg
287 336 29 14
277 318 50 50
DATASET/colored cropped/M08/words/05/07/color_002.jpg
285 334 32 18
276 318 50 50
DATASET/colored cropped/M08/words/05/07/color_003.jpg
285 334 34 16
277 317 50 50
DATASET/colored cropped/M08/words/05/07/color_004.jpg
284 335 35 13
277 317 50 50
DATASET/colored cropped/M08/words/05/07/color_005.jpg
288 337 26 10
276 317 50 50
DATASET/colored cropped/M08/words/05/07/color_006.jpg
288 337 25 10
276 317 50 50
DATASET/colored cropped/M08/words/05/07/color_007.jpg
DATASET/M08/words/05/08/
289 339 24 10
276 319 50 50
DATASET/colored cropped/M08/words/05/08/color_001.jpg
287 337 28 13
276 319 50 50
DATASET/colored cropped/M08/words/05/08/color_002.jpg
285 335 32 18
276 319 50 50
DATASET/colored cropped/M08/words/05/08/color_003.jpg
283 334 36 15
276 317 50 50
DATASET/colored cropp

287 338 27 9
276 318 50 50
DATASET/colored cropped/M08/words/06/08/color_001.jpg
285 339 30 9
275 319 50 50
DATASET/colored cropped/M08/words/06/08/color_002.jpg
283 336 35 15
276 319 50 50
DATASET/colored cropped/M08/words/06/08/color_003.jpg
282 335 36 15
275 318 50 50
DATASET/colored cropped/M08/words/06/08/color_004.jpg
286 338 29 9
276 318 50 50
DATASET/colored cropped/M08/words/06/08/color_005.jpg
288 338 24 12
275 319 50 50
DATASET/colored cropped/M08/words/06/08/color_006.jpg
288 338 25 11
276 319 50 50
DATASET/colored cropped/M08/words/06/08/color_007.jpg
288 337 24 12
275 318 50 50
DATASET/colored cropped/M08/words/06/08/color_008.jpg
288 338 25 11
276 319 50 50
DATASET/colored cropped/M08/words/06/08/color_009.jpg
287 339 26 9
275 319 50 50
DATASET/colored cropped/M08/words/06/08/color_010.jpg
DATASET/M08/words/06/09/
287 339 26 10
275 319 50 50
DATASET/colored cropped/M08/words/06/09/color_001.jpg
287 339 26 12
275 320 50 50
DATASET/colored cropped/M08/words/06/09/color_002

284 338 26 15
272 321 50 50
DATASET/colored cropped/M08/words/07/09/color_004.jpg
284 336 27 12
273 317 50 50
DATASET/colored cropped/M08/words/07/09/color_005.jpg
284 336 27 12
273 317 50 50
DATASET/colored cropped/M08/words/07/09/color_006.jpg
283 337 28 10
272 317 50 50
DATASET/colored cropped/M08/words/07/09/color_007.jpg
DATASET/M08/words/07/10/
283 339 27 10
272 319 50 50
DATASET/colored cropped/M08/words/07/10/color_001.jpg
283 339 27 13
272 321 50 50
DATASET/colored cropped/M08/words/07/10/color_002.jpg
283 339 26 11
271 320 50 50
DATASET/colored cropped/M08/words/07/10/color_003.jpg
284 338 25 14
272 320 50 50
DATASET/colored cropped/M08/words/07/10/color_004.jpg
284 336 26 17
272 320 50 50
DATASET/colored cropped/M08/words/07/10/color_005.jpg
284 337 26 12
272 318 50 50
DATASET/colored cropped/M08/words/07/10/color_006.jpg
283 337 28 10
272 317 50 50
DATASET/colored cropped/M08/words/07/10/color_007.jpg
DATASET/M08/words/08/01/
280 342 27 10
269 322 50 50
DATASET/colored crop

287 327 35 20
280 312 50 50
DATASET/colored cropped/M08/words/09/01/color_003.jpg
291 329 26 16
279 312 50 50
DATASET/colored cropped/M08/words/09/01/color_004.jpg
293 331 22 12
279 312 50 50
DATASET/colored cropped/M08/words/09/01/color_005.jpg
293 332 22 10
279 312 50 50
DATASET/colored cropped/M08/words/09/01/color_006.jpg
293 333 23 10
280 313 50 50
DATASET/colored cropped/M08/words/09/01/color_007.jpg
292 333 25 9
280 313 50 50
DATASET/colored cropped/M08/words/09/01/color_008.jpg
290 332 28 9
279 312 50 50
DATASET/colored cropped/M08/words/09/01/color_009.jpg
DATASET/M08/words/09/02/
288 332 30 9
278 312 50 50
DATASET/colored cropped/M08/words/09/02/color_001.jpg
288 330 33 16
280 313 50 50
DATASET/colored cropped/M08/words/09/02/color_002.jpg
287 328 35 20
280 313 50 50
DATASET/colored cropped/M08/words/09/02/color_003.jpg
287 328 35 19
280 313 50 50
DATASET/colored cropped/M08/words/09/02/color_004.jpg
291 329 27 17
280 313 50 50
DATASET/colored cropped/M08/words/09/02/color_00

294 330 24 10
281 310 50 50
DATASET/colored cropped/M08/words/09/10/color_010.jpg
DATASET/M08/words/10/01/
287 335 24 9
274 315 50 50
DATASET/colored cropped/M08/words/10/01/color_001.jpg
288 335 22 11
274 316 50 50
DATASET/colored cropped/M08/words/10/01/color_002.jpg
286 334 28 13
275 316 50 50
DATASET/colored cropped/M08/words/10/01/color_003.jpg
286 334 28 9
275 314 50 50
DATASET/colored cropped/M08/words/10/01/color_004.jpg
286 334 27 10
275 314 50 50
DATASET/colored cropped/M08/words/10/01/color_005.jpg
287 334 27 8
276 313 50 50
DATASET/colored cropped/M08/words/10/01/color_006.jpg
DATASET/M08/words/10/02/
288 332 25 9
276 312 50 50
DATASET/colored cropped/M08/words/10/02/color_001.jpg
289 333 22 11
275 314 50 50
DATASET/colored cropped/M08/words/10/02/color_002.jpg
288 330 27 18
277 314 50 50
DATASET/colored cropped/M08/words/10/02/color_003.jpg
289 333 24 8
276 312 50 50
DATASET/colored cropped/M08/words/10/02/color_004.jpg
288 332 25 9
276 312 50 50
DATASET/colored cropped/M0

In [13]:
dim = (227, 227)

for person_ID in people:
	if not os.path.exists('DATASET/Resized/'+ person_ID):
		os.mkdir('DATASET/Resized/'+ person_ID)

	for data_type in data_types:
		if not os.path.exists('DATASET/Resized/' + person_ID + '/' + data_type):
			os.mkdir('DATASET/Resized/' + person_ID + '/' + data_type)

		for phrase_ID in folder_enum:
			if not os.path.exists('DATASET/Resized/' + person_ID + '/' + data_type + '/' + phrase_ID):
		         # F01/phrases/01
				os.mkdir('DATASET/Resized/' + person_ID + '/' + data_type + '/' + phrase_ID)
			for instance_ID in instances:
				directory = 'DATASET/colored cropped/' + person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID + '/'
				dir_temp = person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID + '/'
				filelist = os.listdir(directory)
				if not os.path.exists('DATASET/Resized/' + person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID):
					os.mkdir('DATASET/Resized/' + person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID)
				for img_name in filelist:
					if img_name.startswith('color'):
						img = cv2.imread(directory+img_name, cv2.IMREAD_UNCHANGED)
						resized = cv2.resize(img, dim, interpolation = cv2.INTER_AREA)
						cv2.imwrite('DATASET/Resized/'+dir_temp+'/'+img_name,resized)
						print('DATASET/Resized/'+dir_temp+'/'+img_name)

DATASET/Resized/F01/words/01/01//color_001.jpg
DATASET/Resized/F01/words/01/01//color_002.jpg
DATASET/Resized/F01/words/01/01//color_003.jpg
DATASET/Resized/F01/words/01/01//color_004.jpg
DATASET/Resized/F01/words/01/01//color_005.jpg
DATASET/Resized/F01/words/01/01//color_006.jpg
DATASET/Resized/F01/words/01/01//color_007.jpg
DATASET/Resized/F01/words/01/01//color_008.jpg
DATASET/Resized/F01/words/01/01//color_009.jpg
DATASET/Resized/F01/words/01/01//color_010.jpg
DATASET/Resized/F01/words/01/02//color_001.jpg
DATASET/Resized/F01/words/01/02//color_002.jpg
DATASET/Resized/F01/words/01/02//color_003.jpg
DATASET/Resized/F01/words/01/02//color_004.jpg
DATASET/Resized/F01/words/01/02//color_005.jpg
DATASET/Resized/F01/words/01/02//color_006.jpg
DATASET/Resized/F01/words/01/02//color_007.jpg
DATASET/Resized/F01/words/01/03//color_001.jpg
DATASET/Resized/F01/words/01/03//color_002.jpg
DATASET/Resized/F01/words/01/03//color_003.jpg
DATASET/Resized/F01/words/01/03//color_004.jpg
DATASET/Resiz

DATASET/Resized/F01/words/03/08//color_001.jpg
DATASET/Resized/F01/words/03/08//color_002.jpg
DATASET/Resized/F01/words/03/08//color_003.jpg
DATASET/Resized/F01/words/03/08//color_004.jpg
DATASET/Resized/F01/words/03/08//color_005.jpg
DATASET/Resized/F01/words/03/08//color_006.jpg
DATASET/Resized/F01/words/03/08//color_007.jpg
DATASET/Resized/F01/words/03/08//color_008.jpg
DATASET/Resized/F01/words/03/08//color_009.jpg
DATASET/Resized/F01/words/03/08//color_010.jpg
DATASET/Resized/F01/words/03/09//color_001.jpg
DATASET/Resized/F01/words/03/09//color_002.jpg
DATASET/Resized/F01/words/03/09//color_003.jpg
DATASET/Resized/F01/words/03/09//color_004.jpg
DATASET/Resized/F01/words/03/09//color_005.jpg
DATASET/Resized/F01/words/03/09//color_006.jpg
DATASET/Resized/F01/words/03/09//color_007.jpg
DATASET/Resized/F01/words/03/09//color_008.jpg
DATASET/Resized/F01/words/03/09//color_009.jpg
DATASET/Resized/F01/words/03/09//color_010.jpg
DATASET/Resized/F01/words/03/10//color_001.jpg
DATASET/Resiz

DATASET/Resized/F01/words/05/09//color_007.jpg
DATASET/Resized/F01/words/05/10//color_001.jpg
DATASET/Resized/F01/words/05/10//color_002.jpg
DATASET/Resized/F01/words/05/10//color_003.jpg
DATASET/Resized/F01/words/05/10//color_004.jpg
DATASET/Resized/F01/words/05/10//color_005.jpg
DATASET/Resized/F01/words/05/10//color_006.jpg
DATASET/Resized/F01/words/05/10//color_007.jpg
DATASET/Resized/F01/words/05/10//color_008.jpg
DATASET/Resized/F01/words/06/01//color_001.jpg
DATASET/Resized/F01/words/06/01//color_002.jpg
DATASET/Resized/F01/words/06/01//color_003.jpg
DATASET/Resized/F01/words/06/01//color_004.jpg
DATASET/Resized/F01/words/06/01//color_005.jpg
DATASET/Resized/F01/words/06/01//color_006.jpg
DATASET/Resized/F01/words/06/01//color_007.jpg
DATASET/Resized/F01/words/06/01//color_008.jpg
DATASET/Resized/F01/words/06/01//color_009.jpg
DATASET/Resized/F01/words/06/02//color_001.jpg
DATASET/Resized/F01/words/06/02//color_002.jpg
DATASET/Resized/F01/words/06/02//color_003.jpg
DATASET/Resiz

DATASET/Resized/F01/words/08/07//color_001.jpg
DATASET/Resized/F01/words/08/07//color_002.jpg
DATASET/Resized/F01/words/08/07//color_003.jpg
DATASET/Resized/F01/words/08/07//color_004.jpg
DATASET/Resized/F01/words/08/07//color_005.jpg
DATASET/Resized/F01/words/08/07//color_006.jpg
DATASET/Resized/F01/words/08/07//color_007.jpg
DATASET/Resized/F01/words/08/07//color_008.jpg
DATASET/Resized/F01/words/08/08//color_001.jpg
DATASET/Resized/F01/words/08/08//color_002.jpg
DATASET/Resized/F01/words/08/08//color_003.jpg
DATASET/Resized/F01/words/08/08//color_004.jpg
DATASET/Resized/F01/words/08/08//color_005.jpg
DATASET/Resized/F01/words/08/08//color_006.jpg
DATASET/Resized/F01/words/08/08//color_007.jpg
DATASET/Resized/F01/words/08/08//color_008.jpg
DATASET/Resized/F01/words/08/09//color_001.jpg
DATASET/Resized/F01/words/08/09//color_002.jpg
DATASET/Resized/F01/words/08/09//color_003.jpg
DATASET/Resized/F01/words/08/09//color_004.jpg
DATASET/Resized/F01/words/08/09//color_005.jpg
DATASET/Resiz

DATASET/Resized/F02/words/02/01//color_001.jpg
DATASET/Resized/F02/words/02/01//color_002.jpg
DATASET/Resized/F02/words/02/01//color_003.jpg
DATASET/Resized/F02/words/02/01//color_004.jpg
DATASET/Resized/F02/words/02/01//color_005.jpg
DATASET/Resized/F02/words/02/01//color_006.jpg
DATASET/Resized/F02/words/02/01//color_007.jpg
DATASET/Resized/F02/words/02/02//color_001.jpg
DATASET/Resized/F02/words/02/02//color_002.jpg
DATASET/Resized/F02/words/02/02//color_003.jpg
DATASET/Resized/F02/words/02/02//color_004.jpg
DATASET/Resized/F02/words/02/02//color_005.jpg
DATASET/Resized/F02/words/02/02//color_006.jpg
DATASET/Resized/F02/words/02/02//color_007.jpg
DATASET/Resized/F02/words/02/02//color_008.jpg
DATASET/Resized/F02/words/02/02//color_009.jpg
DATASET/Resized/F02/words/02/03//color_001.jpg
DATASET/Resized/F02/words/02/03//color_002.jpg
DATASET/Resized/F02/words/02/03//color_003.jpg
DATASET/Resized/F02/words/02/03//color_004.jpg
DATASET/Resized/F02/words/02/03//color_005.jpg
DATASET/Resiz

...

DATASET/Resized/M07/words/09/02//color_005.jpg
DATASET/Resized/M07/words/09/02//color_006.jpg
DATASET/Resized/M07/words/09/02//color_007.jpg
DATASET/Resized/M07/words/09/02//color_008.jpg
DATASET/Resized/M07/words/09/03//color_001.jpg
DATASET/Resized/M07/words/09/03//color_002.jpg
DATASET/Resized/M07/words/09/03//color_003.jpg
DATASET/Resized/M07/words/09/03//color_004.jpg
DATASET/Resized/M07/words/09/03//color_005.jpg
DATASET/Resized/M07/words/09/03//color_006.jpg
DATASET/Resized/M07/words/09/03//color_007.jpg
DATASET/Resized/M07/words/09/03//color_008.jpg
DATASET/Resized/M07/words/09/04//color_001.jpg
DATASET/Resized/M07/words/09/04//color_002.jpg
DATASET/Resized/M07/words/09/04//color_003.jpg
DATASET/Resized/M07/words/09/04//color_004.jpg
DATASET/Resized/M07/words/09/04//color_005.jpg
DATASET/Resized/M07/words/09/04//color_006.jpg
DATASET/Resized/M07/words/09/04//color_007.jpg
DATASET/Resized/M07/words/09/04//color_008.jpg
DATASET/Resized/M07/words/09/05//color_001.jpg
DATASET/Resiz

DATASET/Resized/M08/words/01/10//color_004.jpg
DATASET/Resized/M08/words/01/10//color_005.jpg
DATASET/Resized/M08/words/01/10//color_006.jpg
DATASET/Resized/M08/words/01/10//color_007.jpg
DATASET/Resized/M08/words/01/10//color_008.jpg
DATASET/Resized/M08/words/02/01//color_001.jpg
DATASET/Resized/M08/words/02/01//color_002.jpg
DATASET/Resized/M08/words/02/01//color_003.jpg
DATASET/Resized/M08/words/02/01//color_004.jpg
DATASET/Resized/M08/words/02/01//color_005.jpg
DATASET/Resized/M08/words/02/01//color_006.jpg
DATASET/Resized/M08/words/02/01//color_007.jpg
DATASET/Resized/M08/words/02/01//color_008.jpg
DATASET/Resized/M08/words/02/01//color_009.jpg
DATASET/Resized/M08/words/02/01//color_010.jpg
DATASET/Resized/M08/words/02/02//color_001.jpg
DATASET/Resized/M08/words/02/02//color_002.jpg
DATASET/Resized/M08/words/02/02//color_003.jpg
DATASET/Resized/M08/words/02/02//color_004.jpg
DATASET/Resized/M08/words/02/02//color_005.jpg
DATASET/Resized/M08/words/02/02//color_006.jpg
DATASET/Resiz

DATASET/Resized/M08/words/04/07//color_004.jpg
DATASET/Resized/M08/words/04/07//color_005.jpg
DATASET/Resized/M08/words/04/07//color_006.jpg
DATASET/Resized/M08/words/04/07//color_007.jpg
DATASET/Resized/M08/words/04/07//color_008.jpg
DATASET/Resized/M08/words/04/07//color_009.jpg
DATASET/Resized/M08/words/04/08//color_001.jpg
DATASET/Resized/M08/words/04/08//color_002.jpg
DATASET/Resized/M08/words/04/08//color_003.jpg
DATASET/Resized/M08/words/04/08//color_004.jpg
DATASET/Resized/M08/words/04/08//color_005.jpg
DATASET/Resized/M08/words/04/08//color_006.jpg
DATASET/Resized/M08/words/04/08//color_007.jpg
DATASET/Resized/M08/words/04/08//color_008.jpg
DATASET/Resized/M08/words/04/08//color_009.jpg
DATASET/Resized/M08/words/04/09//color_001.jpg
DATASET/Resized/M08/words/04/09//color_002.jpg
DATASET/Resized/M08/words/04/09//color_003.jpg
DATASET/Resized/M08/words/04/09//color_004.jpg
DATASET/Resized/M08/words/04/09//color_005.jpg
DATASET/Resized/M08/words/04/09//color_006.jpg
DATASET/Resiz

DATASET/Resized/M08/words/07/03//color_008.jpg
DATASET/Resized/M08/words/07/04//color_001.jpg
DATASET/Resized/M08/words/07/04//color_002.jpg
DATASET/Resized/M08/words/07/04//color_003.jpg
DATASET/Resized/M08/words/07/04//color_004.jpg
DATASET/Resized/M08/words/07/04//color_005.jpg
DATASET/Resized/M08/words/07/04//color_006.jpg
DATASET/Resized/M08/words/07/04//color_007.jpg
DATASET/Resized/M08/words/07/05//color_001.jpg
DATASET/Resized/M08/words/07/05//color_002.jpg
DATASET/Resized/M08/words/07/05//color_003.jpg
DATASET/Resized/M08/words/07/05//color_004.jpg
DATASET/Resized/M08/words/07/05//color_005.jpg
DATASET/Resized/M08/words/07/05//color_006.jpg
DATASET/Resized/M08/words/07/05//color_007.jpg
DATASET/Resized/M08/words/07/06//color_001.jpg
DATASET/Resized/M08/words/07/06//color_002.jpg
DATASET/Resized/M08/words/07/06//color_003.jpg
DATASET/Resized/M08/words/07/06//color_004.jpg
DATASET/Resized/M08/words/07/06//color_005.jpg
DATASET/Resized/M08/words/07/06//color_006.jpg
DATASET/Resiz

DATASET/Resized/M08/words/09/07//color_007.jpg
DATASET/Resized/M08/words/09/07//color_008.jpg
DATASET/Resized/M08/words/09/07//color_009.jpg
DATASET/Resized/M08/words/09/07//color_010.jpg
DATASET/Resized/M08/words/09/07//color_011.jpg
DATASET/Resized/M08/words/09/08//color_001.jpg
DATASET/Resized/M08/words/09/08//color_002.jpg
DATASET/Resized/M08/words/09/08//color_003.jpg
DATASET/Resized/M08/words/09/08//color_004.jpg
DATASET/Resized/M08/words/09/08//color_005.jpg
DATASET/Resized/M08/words/09/08//color_006.jpg
DATASET/Resized/M08/words/09/08//color_007.jpg
DATASET/Resized/M08/words/09/08//color_008.jpg
DATASET/Resized/M08/words/09/08//color_009.jpg
DATASET/Resized/M08/words/09/09//color_001.jpg
DATASET/Resized/M08/words/09/09//color_002.jpg
DATASET/Resized/M08/words/09/09//color_003.jpg
DATASET/Resized/M08/words/09/09//color_004.jpg
DATASET/Resized/M08/words/09/09//color_005.jpg
DATASET/Resized/M08/words/09/09//color_006.jpg
DATASET/Resized/M08/words/09/09//color_007.jpg
DATASET/Resiz

In [29]:
img=cv2.imread('DATASET/colored cropped/M02/words/01//01/color_009.jpg',cv2.IMREAD_UNCHANGED)
print(img.shape)

(100, 100, 3)


In [15]:
path='DATASET/Resized/'
dir_list = os.listdir(path) 
print("Files and directories in '", path, "' :")  
l=[]

Files and directories in ' DATASET/Resized/ ' :


In [16]:
def senditems(dir_list,person_ID,phrase_ID,instance_ID):
	img=[]
	for i in dir_list:
		img1=cv2.imread(i)
		img.append(img1)
	n=25
	total=25
	len_of_l=len(img)
	itt=0
	while (len_of_l<=n):
		itt=itt+1
		n=n-len_of_l
	seq=[]
	diff=[]
	for i in range(0,(len(img)-1)):
		minus = np.sum(np.abs(np.subtract(img[i],img[i+1],dtype=np.float)))
		diff.append(minus)
	diff1=[]
	for i in diff:
		diff1.append(i)
	diff1.sort(reverse=True)
	for i in range(0,n):
		seq.append(diff.index(diff1[i])+1)
	for i in range(0,len(img)):
		for j in range(0,itt):
			seq.append(i)
	seq.sort()
	print(seq)

In [17]:
def senditems(dir_list,person_ID,phrase_ID,instance_ID):
	img=[]
	for i in dir_list:
		img1=cv2.imread(i)
		img.append(img1)
	n=25
	total=25
	len_of_l=len(img)
	itt=0
	while (len_of_l<=n):
		itt=itt+1
		n=n-len_of_l
	seq=[]
	diff=[]
	for i in range(0,(len(img)-1)):
		minus = np.sum(np.abs(np.subtract(img[i],img[i+1],dtype=np.float)))
		diff.append(minus)
	diff1=[]
	for i in diff:
		diff1.append(i)
	diff1.sort(reverse=True)
	for i in range(0,n):
		seq.append(diff.index(diff1[i])+1)
	for i in range(0,len(img)):
		for j in range(0,itt):
			seq.append(i)
	seq.sort()
	print(seq)

	def concat_tile(im_list_2d):
		return cv2.vconcat([cv2.hconcat(im_list_h) for im_list_h in im_list_2d])

	im_tile = concat_tile([[img[seq[0]],img[seq[1]],img[seq[2]],img[seq[3]],img[seq[4]]],
												[img[seq[5]],img[seq[6]],img[seq[7]],img[seq[8]],img[seq[9]]],
												[img[seq[10]],img[seq[11]],img[seq[12]],img[seq[13]],img[seq[14]]],
												[img[seq[15]],img[seq[16]],img[seq[17]],img[seq[18]],img[seq[19]]],
												[img[seq[20]],img[seq[21]],img[seq[22]],img[seq[23]],img[seq[24]]]])
	path='DATASET/Final/'+phrase_ID+'/'+person_ID+'-'+phrase_ID+'-'+instance_ID+'.jpg'
	print(path)
	cv2.imwrite(path, im_tile)
	img=cv2.imread(path)
	print(img.shape)

In [18]:
people = ['F01','F02','F05','F04','F06','F07','F08','F09','F10','F11','M01','M02','M04','M07','M08']
data_types = ['words']
folder_enum = ['01','02','03','04','05','06','07','08', '09', '10']
instances = ['01','02','03','04','05','06','07','08', '09', '10']
words = ['Begin', 'Choose', 'Connection', 'Navigation', 'Next', 'Previous', 'Start', 'Stop', 'Hello', 'Web'] 

#if not os.path.exists('/content/drive/My Drive/Data sets/lip reading/try/colored data-set/M2'):
#    os.mkdir('/content/drive/My Drive/Data sets/lip reading/try/colored data-set/M2')

In [20]:
for person_ID in people:
	for data_type in data_types:
		# if not os.path.exists('/content/drive/My Drive/Data sets/lip reading/try/dataset/' + data_type):#
		# os.mkdir('/content/drive/My Drive/Data sets/lip reading/try/dataset/' + data_type)
		for phrase_ID in folder_enum:
			if not os.path.exists('DATASET/Final/' + phrase_ID ):
				os.mkdir('DATASET/Final/' + phrase_ID)
			# if not os.path.exists('/content/drive/My Drive/Data sets/lip reading/try/colored data-set/M2/' + phrase_ID):
			# F01/phrases/01
			# os.mkdir('/content/drive/My Drive/Data sets/lip reading/try/colored data-set/M2/' + phrase_ID)
			for instance_ID in instances:
				# F01/phrases/01/01
				directory = 'DATASET/Resized/' + person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID + '/'
				dir_temp = person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID + '/'
				#print(directory)
				filelist = os.listdir(directory)
				# if not os.path.exists('cropped/' + person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID):#
				# os.mkdir('/content/drive/My Drive/Data sets/lip reading/try/cropped/' + person_ID + '/' + data_type + '/' + phrase_ID + '/' + instance_ID)
				p=[]
				for img_name in filelist:
					p.append(directory+img_name)
				p.sort()
				senditems(p,person_ID,phrase_ID,instance_ID)
  


C:\Users\Dell\AppData\Local\Temp\ipykernel_11232\2278849221.py:16: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  minus = np.sum(np.abs(np.subtract(img[i],img[i+1],dtype=np.float)))


[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 9, 9]
DATASET/Final/01/F01-01-01.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 6, 6, 6]
DATASET/Final/01/F01-01-02.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 7, 7, 7, 8, 8, 8, 9, 9]
DATASET/Final/01/F01-01-03.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8]
DATASET/Final/01/F01-01-04.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 9, 10, 10]
DATASET/Final/01/F01-01-05.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/01/F01-01-06.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 7]
DATASET/Final/01/F01-01-07.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7]
DATASET/Final/01/F01-01-08.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1

[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 6, 6, 6]
DATASET/Final/07/F01-07-09.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 8, 8, 8]
DATASET/Final/07/F01-07-10.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8, 9, 9]
DATASET/Final/08/F01-08-01.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 6]
DATASET/Final/08/F01-08-02.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 6]
DATASET/Final/08/F01-08-03.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7]
DATASET/Final/08/F01-08-04.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 6]
DATASET/Final/08/F01-08-05.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 6, 6, 6]
DATASET/Final/08/F01-08-06.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 

[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 8, 8, 8]
DATASET/Final/04/F02-04-08.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7]
DATASET/Final/04/F02-04-09.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7]
DATASET/Final/04/F02-04-10.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 5, 6, 6, 6, 7, 7, 7]
DATASET/Final/05/F02-05-01.jpg
(1135, 1135, 3)
[0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5]
DATASET/Final/05/F02-05-02.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/05/F02-05-03.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 6]
DATASET/Final/05/F02-05-04.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7]
DATASET/Final/05/F02-05-05.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 

[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12]
DATASET/Final/01/F05-01-05.jpg
(1135, 1135, 3)
[0, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12, 13]
DATASET/Final/01/F05-01-06.jpg
(1135, 1135, 3)
[0, 1, 1, 2, 2, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 10, 11, 12, 12, 13, 13, 14, 14]
DATASET/Final/01/F05-01-07.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 7, 8, 8, 8, 9, 9, 10, 10]
DATASET/Final/01/F05-01-08.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10]
DATASET/Final/01/F05-01-09.jpg
(1135, 1135, 3)
[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12]
DATASET/Final/01/F05-01-10.jpg
(1135, 1135, 3)
[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12]
DATASET/Final/02/F05-02-01.jpg
(1135, 1135, 3)
[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 8, 8, 9, 9, 10, 11, 11, 12, 12, 13, 13]
DATASET/Final/02/F0

...

[0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 7, 8, 8, 8, 9, 9, 9, 10, 10]
DATASET/Final/02/M04-02-03.jpg
(1135, 1135, 3)
[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12]
DATASET/Final/02/M04-02-04.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 9, 9]
DATASET/Final/02/M04-02-05.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 7, 8, 8, 8, 9, 9, 10, 10]
DATASET/Final/02/M04-02-06.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/02/M04-02-07.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 9, 10, 10, 10]
DATASET/Final/02/M04-02-08.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/02/M04-02-09.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 7, 8, 8, 8, 9, 9, 9]
DATASET/Final/02/M04-02-10.jpg
(1135, 1135, 3)
[0,

[0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11]
DATASET/Final/08/M04-08-08.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10]
DATASET/Final/08/M04-08-09.jpg
(1135, 1135, 3)
[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12]
DATASET/Final/08/M04-08-10.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10]
DATASET/Final/09/M04-09-01.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 8, 9, 9]
DATASET/Final/09/M04-09-02.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 9, 9]
DATASET/Final/09/M04-09-03.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 10]
DATASET/Final/09/M04-09-04.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10]
DATASET/Final/09/M04-09-05.jpg
(1135, 1135, 

[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/05/M07-05-05.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/05/M07-05-06.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8]
DATASET/Final/05/M07-05-07.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/05/M07-05-08.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/05/M07-05-09.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/05/M07-05-10.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 8, 8, 9, 9]
DATASET/Final/06/M07-06-01.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 4, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 9, 9, 10, 10]
DATASET/Final/06/M07-06-02.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1

[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/02/M08-02-03.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/02/M08-02-04.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 6]
DATASET/Final/02/M08-02-05.jpg
(1135, 1135, 3)
[0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7]
DATASET/Final/02/M08-02-06.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10]
DATASET/Final/02/M08-02-07.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 6, 6, 7, 7, 7, 8, 8, 8, 9, 9]
DATASET/Final/02/M08-02-08.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/02/M08-02-09.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 8, 9, 9, 9]
DATASET/Final/02/M08-02-10.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2

[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 8, 8, 8]
DATASET/Final/09/M08-09-01.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 8, 8, 9, 9]
DATASET/Final/09/M08-09-02.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 8, 9, 9]
DATASET/Final/09/M08-09-03.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 6, 6, 6, 7, 7, 8, 8, 9, 9, 9]
DATASET/Final/09/M08-09-04.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9]
DATASET/Final/09/M08-09-05.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 7, 7, 7, 8, 8, 8]
DATASET/Final/09/M08-09-06.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 9, 10, 10]
DATASET/Final/09/M08-09-07.jpg
(1135, 1135, 3)
[0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 8, 8, 8]
DATASET/Final/09/M08-09-08.jpg
(1135, 1135, 3)
[0, 1, 1, 2, 2